# Normalisasi Scopus — deteksi duplikat, validasi DOI, merge, dan tabel bidang
Membaca **beberapa CSV Scopus** (satu file = satu bidang ilmu) lalu menghasilkan tabel relasional
(PostgreSQL-ready) **tanpa duplikat publikasi**.

**Prinsip: 100% otomatis, tanpa pemeriksaan manual.** Dua record digabung **hanya jika ketiga syarat terpenuhi, berurutan:
(1) DOI, (2) judul, (3) author.** EID **bukan** patokan merge (record duplikat bisa punya EID berbeda). Kalau ada syarat yang tidak
terpenuhi, record **tidak digabung** dan alasannya dicatat di `log_keputusan_duplikat.csv` + sheet `PASANGAN_TIDAK_DIGABUNG`.
DOI yang tercatat diperiksa otomatis (bisa dibuka & cocok dengan judul/tahun/author); DOI yang salah dicarikan DOI yang benar
lewat Crossref, dan bila tidak ada diganti EID. DOI tidak pernah kosong.

| Tahap | Isi |
|---|---|
| 1 | Baca semua file + simpan identitas file sumber + cadangan data mentah |
| 2 | Pemeriksaan struktur & kualitas data |
| 3 | Deteksi kandidat duplikat **dalam satu file** |
| 4 | Validasi DOI kandidat (format → bisa diakses → cocok dengan metadata) |
| 5 | Merge duplikat dalam file yang sudah terkonfirmasi |
| 6 | Deteksi kandidat duplikat **antar-file** |
| 7 | Validasi DOI kandidat lintas file |
| 8 | Merge antar-file yang terbukti sama |
| 9 | `publication_id` berurutan tanpa celah |
| 10 | Tabel `bidang` dan `pub_bidang` |
| 11 | Tabel keseluruhan + tabel per CSV |
| 12 | Laporan akhir + log |

**Aturan penentuan publikasi yang sama** (dicek berurutan): EID sama → DOI sama *dan tervalidasi* → judul + author sama
(`AMBANG_AUTHOR_SAMA`). Jika ada DOI, minimal satu DOI harus valid dan cocok dengan metadata sebelum merge.
Tidak ada ID berakhiran `-D1`: publikasi yang sama = **satu** `publication_id`, dan **semua bidang asalnya** tercatat di `pub_bidang`.

**ID dihitung ulang dari awal setiap Run** (urutan: file di `DAFTAR_FILE` dari atas ke bawah → urutan baris),
jadi proses aman dijalankan berulang dan ID selalu rapat tanpa celah. Menambah file di *akhir* daftar tidak mengubah ID yang ada;
menghapus / menyisipkan file di tengah menggeser ID sesudahnya (perubahan dicatat di `log_perubahan_id.csv`).

Cara pakai: ubah pengaturan di sel berikutnya → **Runtime → Run all**.

In [1]:
# ============ PENGATURAN (ubah bagian ini saja) ============
NAMA_FOLDER = "NORMALISASI DATA BRIN"     # folder di My Drive berisi SEMUA file CSV

# ---- FILE YANG DIPROSES: OTOMATIS dari isi folder ----
# Cukup UPLOAD file CSV baru ke folder di atas lalu Run all: semua CSV di folder otomatis terbaca. Tidak perlu menulis nama file.
# Urutan (menentukan urutan ID): file yang sudah diproses di Run sebelumnya tetap di urutan lamanya (ID-nya tidak bergeser),
# file BARU ditaruh di paling akhir (urut waktu upload). Nama bidang = nama file tanpa ".csv".
# Nama bidang yang rapi (opsional, hanya untuk file yang perlu): isi {"nama file.csv": "Nama Bidang"}.
BIDANG_KHUSUS = {
    # "Physics-and-Astronomy-ver-1.csv": "Physics",
    # "Medicine.csv": "Medicine",
}
# (Cara lama, opsional) Isi DAFTAR_FILE untuk menentukan sendiri file, urutan, dan bidang. Kosong [] = otomatis dari folder.
DAFTAR_FILE = [
    # {"file": "Physics.csv",   "bidang": "Physics"},
    # {"file": "Medicine.csv",  "bidang": "Medicine"},
]
# Jika DAFTAR_FILE dikosongkan ( [] ): SEMUA CSV di folder diproses (lihat aturan urutan di atas).

# Data awal / file acuan: TIDAK dipakai (None) -> file pertama di DAFTAR_FILE mulai dari PUB000001.
# (Jika suatu saat ingin memakai prototype 20 baris lagi: isi "scopus_research_intelligence_prototype.xlsx" dan taruh di folder yang sama.)
FILE_ACUAN = None

NAMA_FOLDER_HASIL = "hasil_normalisasi"      # dibuat otomatis di dalam NAMA_FOLDER
SIMPAN_HASIL = True                          # False = coba-coba (dry run): semua tahap jalan & laporan tampil, tetapi TIDAK ada file ditulis

# ---- Validasi DOI ----
VALIDASI_DOI = "semua"      # "semua" = semua DOI dicek otomatis (default) | "kandidat" = hanya DOI yang terlibat kandidat duplikat (cepat) | "mati" = tanpa validasi online
WAJIB_DOI_VALID = True      # True = DOI baru boleh jadi dasar merge jika sudah tervalidasi (bisa dibuka + cocok dengan metadata)
AMBANG_JUDUL_DOI = 0.90     # kemiripan judul di CSV vs judul pada metadata DOI (0-1): >= nilai ini cukup untuk dianggap cocok
AMBANG_JUDUL_LONGGAR = 0.70 # kemiripan judul minimal bila tahun (+-1) DAN salah satu author juga cocok (judul dengan rumus/simbol sering berbeda penulisan)
DOI_TIMEOUT = 20            # detik per permintaan
DOI_PARALEL = 6             # jumlah permintaan DOI bersamaan (jangan terlalu besar: server membatasi -> status tidak_dapat_diakses)
DOI_COBA_ULANG = 4          # percobaan ulang (jeda makin lama) bila server sibuk / membatasi permintaan / timeout
DOI_EMAIL = ""              # (opsional) email Anda; Crossref memberi jalur lebih longgar bila ada. Kosong = tidak dikirim

# ---- Kemiripan untuk merge tanpa EID/DOI yang sama ----
AMBANG_AUTHOR_SAMA = 0.8    # minimal sekian bagian author sama (cocok lewat Scopus Author ID atau nama)
AMBANG_JUDUL_SAMA = 0.95    # judul dianggap sama bila hasil normalisasi identik ATAU kemiripannya >= nilai ini (kemiripan hanya untuk pasangan ber-DOI sama)
PAKSA_GABUNG = []           # (opsional, jarang perlu) paksa 2 record = 1 publikasi, mis. [("2-s2.0-111", "2-s2.0-222")] (pakai EID)
KOREKSI_DOI = True          # True = DOI yang salah (tidak cocok / tidak ditemukan / format salah) dicarikan DOI yang benar lewat Crossref; bila tidak ada diganti EID
PILIH_BARIS_UTAMA = "terlengkap"   # record utama saat merge: "terlengkap" (kolom terisi terbanyak) | "pertama" (paling atas)

MAKS_KARAKTER_CHUNK = 1200   # panjang maksimum 1 chunk (title+abstract dipotong di akhir kalimat)
MODE_REFERENCES     = "smart"   # "smart" = 1 referensi utuh per baris | "naive" = pisah tiap ';'
SIMPAN_KE_SQLITE = False     # True = simpan juga database SQLite (opsional)
# ===========================================================

import os, re, io, json, time, shutil, hashlib, unicodedata, threading, html
from datetime import datetime
from difflib import SequenceMatcher
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import quote
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{NAMA_FOLDER}"
except ImportError:                       # di luar Colab
    BASE_DIR = os.environ.get("BASE_DIR", ".")

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"Folder tidak ditemukan: {BASE_DIR}")
HASIL_DIR   = os.path.join(BASE_DIR, NAMA_FOLDER_HASIL)
LAPORAN_DIR = os.path.join(HASIL_DIR, "laporan")
BACKUP_DIR  = os.path.join(HASIL_DIR, "data_mentah_backup")
FILE_CACHE_DOI = os.path.join(LAPORAN_DIR, "cache_validasi_doi.csv")
FILE_PEMETAAN  = os.path.join(LAPORAN_DIR, "pemetaan_publication_id.csv")

# ---- daftar file yang akan diproses ----
_csv_di_folder = sorted(f for f in os.listdir(BASE_DIR) if f.lower().endswith(".csv"))
if not DAFTAR_FILE:
    # urutan otomatis: yang sudah pernah diproses (urutan Run sebelumnya) dulu, file baru di akhir menurut waktu upload
    _urut_lama = []
    if os.path.exists(FILE_PEMETAAN):
        _urut_lama = list(dict.fromkeys(pd.read_csv(FILE_PEMETAAN, dtype=str, encoding="utf-8-sig")["file_sumber"]))
    _lama = [f for f in _urut_lama if f in _csv_di_folder]
    _baru = sorted((f for f in _csv_di_folder if f not in _lama), key=lambda f: (os.path.getmtime(os.path.join(BASE_DIR, f)), f))
    _daftar = [{"file": f} for f in _lama + _baru]
    if _baru and _lama:
        print("File baru terdeteksi (ditaruh di urutan terakhir):", _baru)
else:
    _daftar = DAFTAR_FILE
FILES = []
for _d in _daftar:
    _d = dict(_d)
    _d.setdefault("bidang", BIDANG_KHUSUS.get(_d["file"], os.path.splitext(_d["file"])[0]))
    FILES.append(_d)
if len(FILES) == 1 and not os.path.exists(os.path.join(BASE_DIR, FILES[0]["file"])) and len(_csv_di_folder) == 1:
    print("Nama file disesuaikan otomatis:", FILES[0]["file"], "->", _csv_di_folder[0])
    FILES[0]["file"] = _csv_di_folder[0]
_hilang = [d["file"] for d in FILES if not os.path.exists(os.path.join(BASE_DIR, d["file"]))]
if _hilang:
    raise FileNotFoundError(f"File tidak ditemukan di {BASE_DIR}: {_hilang}\nIsi folder: {os.listdir(BASE_DIR)}")
ACUAN_PATH = None
if FILE_ACUAN:
    ACUAN_PATH = os.path.join(BASE_DIR, FILE_ACUAN)
    if not os.path.exists(ACUAN_PATH):      # file acuan diminta tetapi tidak ada -> berhenti (jangan diam-diam mulai dari PUB000001)
        raise FileNotFoundError(f"File acuan '{FILE_ACUAN}' tidak ditemukan di {BASE_DIR}.\n"
                                f"Isi folder: {os.listdir(BASE_DIR)}\nJika memang ingin mulai dari PUB000001 tanpa data awal, set FILE_ACUAN = None.")
print(f"{len(FILES)} file CSV akan diproses" + (f" + data awal {FILE_ACUAN}" if ACUAN_PATH else "") + ":")
for _d in FILES:
    print(f"  - {_d['file']}  -> bidang: {_d['bidang']}")

Mounted at /content/drive
9 file CSV akan diproses:
  - AGRICULTURE.csv  -> bidang: AGRICULTURE
  - Computer Science Mathematics Decision Sciences.csv  -> bidang: Computer Science Mathematics Decision Sciences
  - Energy.csv  -> bidang: Energy
  - Ilmu Hayati.csv  -> bidang: Ilmu Hayati
  - Ilmu Lingkungan.csv  -> bidang: Ilmu Lingkungan
  - Ilmu Material.csv  -> bidang: Ilmu Material
  - Medicine.csv  -> bidang: Medicine
  - Physics and Astronomy Earth and Planetary Sciences.csv  -> bidang: Physics and Astronomy Earth and Planetary Sciences
  - Veteriner.csv  -> bidang: Veteriner


## Fungsi bantu (pembersihan teks, kunci pembanding, author, tabel)

In [2]:
LOG = []
def catat(tahap, pesan):
    """Catat ke log proses (disimpan di laporan/log_proses.csv) dan tampilkan."""
    LOG.append({"waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "tahap": tahap, "pesan": pesan})
    print(f"[{tahap}] {pesan}")

def ada(x):
    """True bila x berisi teks (bukan None / NaN / kosong)."""
    return isinstance(x, str) and x != ""

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    x = re.sub(r"[ \t]+", " ", str(x).replace("\r", " ")).strip()
    return None if x == "" or x.lower() == "nan" else x

def split_semicolon(x):
    x = clean_text(x)
    return [p.strip() for p in x.split(";") if p.strip()] if x else []

def snake(col):
    return re.sub(r"[^a-z0-9]+", "_", col.lower()).strip("_")

def make_id(prefix, text, digits=10):
    return f"{prefix}{int(hashlib.md5(text.encode('utf-8')).hexdigest(), 16) % (10 ** digits):0{digits}d}"

def norm_key(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

def judul_key(t):
    """Kunci pembanding judul/nama: huruf kecil, hanya huruf & angka."""
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    return re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", str(t)).casefold())
nama_key = judul_key

def _doi_dasar(d):
    d = clean_text(d)
    return re.sub(r"^(https?://)?(dx\.)?doi\.org/", "", d.strip().lower()).strip() if d else None

def norm_doi(d):
    """DOI bersih: huruf kecil, tanpa URL/'doi:', tanpa spasi, prefix yang terulang (10.7226/10.7226/x -> 10.7226/x) dan tanda baca di ujung dibuang."""
    d = _doi_dasar(d)
    if not d:
        return None
    d = re.sub(r"^doi:\s*", "", d)
    d = re.sub(r"\s+", "", d)
    d = re.sub(r"^(10\.\d{4,9}/)\1", r"\1", d)
    d = d.rstrip(".,;:")
    return d or None


DOI_RE = re.compile(r"^10\.\d{4,9}/\S+$")

KATA_TETAP_KECIL = set()
def capitalize_words(s):
    """Awali tiap kata dengan huruf besar; huruf selanjutnya tidak diubah ('IPB University' tetap)."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return s
    up = lambda m: m.group(0) if m.group(0).lower() in KATA_TETAP_KECIL else m.group(0)[0].upper() + m.group(0)[1:]
    return re.sub(r"[^\W_]+(?:['’][^\W_]+)*", up, s)

def guess_country(name):
    parts = [p.strip() for p in name.split(",")]
    if len(parts) >= 2 and re.fullmatch(r"[A-Za-z .&'\-]{3,40}", parts[-1]):
        return parts[-1].title()
    return None

FULL_RE = re.compile(r"^(?P<name>.*?)\s*\((?P<id>\d+)\)\s*$")

def author_list(r):
    """Daftar author 1 baris: [(scopus_id, kunci nama lengkap, kunci nama singkat), ...]"""
    short = split_semicolon(r.get("authors")); full = split_semicolon(r.get("author_full_names")); ids = split_semicolon(r.get("author_s_id"))
    out = []
    for i in range(max(len(short), len(full), len(ids))):
        m = FULL_RE.match(full[i]) if i < len(full) else None
        fname = m.group("name") if m else (full[i] if i < len(full) else "")
        aid = ids[i] if i < len(ids) else (m.group("id") if m else "")
        out.append((aid, nama_key(fname), nama_key(short[i]) if i < len(short) else ""))
    return out

def rasio_author(A, B):
    """Bagian author yang sama (0..1): cocok bila Scopus ID sama ATAU nama sama."""
    if not A or not B:
        return 0.0
    cocok = sum(any((a[0] and a[0] == b[0]) or (a[1] and a[1] == b[1]) or (a[2] and a[2] == b[2]) for b in B) for a in A)
    return cocok / max(len(A), len(B))

def lengkapi_kolom_turunan(tabel):
    """Kolom turunan: nama institusi berawalan huruf besar, country, dan 3 kolom *_normalized (huruf kecil)."""
    a = tabel["AUTHORS"]
    if len(a):
        a["authors_name_normalized"] = a["author_name"].str.strip().str.lower()
    i = tabel["INSTITUTIONS"]
    if len(i):
        i["institution_name"] = i["institution_name"].map(capitalize_words)
        if "city" not in i: i["city"] = None
        if "country" not in i: i["country"] = None
        i["country"] = i["country"].where(i["country"].notna(), i["institution_name"].map(guess_country))
        i["institutions_name_normalized"] = i["institution_name"].str.strip().str.lower()
    f = tabel["FUNDING"]
    if len(f):
        f["funding_agency_normalized"] = f["funding_agency"].str.strip().str.lower()
    return tabel

def make_chunks(text, limit):
    if not text:
        return []
    bounds = [0] + [m.end() for m in re.finditer(r"(?<=[.!?])\s+|\n\n", text)] + [len(text)]
    segs = [text[a:b] for a, b in zip(bounds, bounds[1:]) if text[a:b].strip()]
    chunks, cur = [], ""
    for sg in segs:
        if cur and len((cur + sg).strip()) > limit:
            chunks.append(cur.strip()); cur = sg
        else:
            cur += sg
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

YEAR_END = re.compile(r"\((?:\d{4}[a-z]?|in press|n\.d\.)\)\s*$", re.I)
def split_references(text, mode):
    parts = split_semicolon(text)
    if mode == "naive":
        return parts
    refs, buf = [], []
    for p in parts:
        buf.append(p)
        if YEAR_END.search(p):
            refs.append("; ".join(buf)); buf = []
    if buf:
        refs.append("; ".join(buf))
    return refs

GRANT_RE = re.compile(r"\(([^()]*\d[^()]*)\)\s*$")
EXCEL_LIMIT = 32000
def safe_for_excel(df):
    cut = lambda v: v[:EXCEL_LIMIT] + " …[dipotong]" if isinstance(v, str) and len(v) > EXCEL_LIMIT else v
    return df.apply(lambda c: c.map(cut) if c.dtype == object or str(c.dtype).startswith("str") else c)

def tulis_excel(path, tabel):
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in tabel.items():
            safe_for_excel(df).to_excel(w, sheet_name=name[:31], index=False)
            ws = w.sheets[name[:31]]
            ws.freeze_panes = "A2"
            for col in ws.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:50])
                ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 10), 60)

def tulis_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")

## Tahap 1 — Baca semua file, simpan identitas file sumber, cadangan data mentah
Tiap baris diberi `file_sumber` + `nomor_baris` (kunci `file_sumber:nomor_baris`), sehingga asal bidangnya tidak hilang.
File asli tidak pernah diubah; salinannya disimpan di `data_mentah_backup/` beserta catatan sha1.

In [3]:
def tahap1_baca():
    sumber = []     # (nama_sumber, bidang, DataFrame mentah)
    prototype = {}
    if ACUAN_PATH:
        sheets = pd.read_excel(ACUAN_PATH, sheet_name=None, dtype=str)
        prototype = sheets
        sumber.append((FILE_ACUAN, None, sheets["RAW"]))
    for d in FILES:
        df = pd.read_csv(os.path.join(BASE_DIR, d["file"]), encoding="utf-8-sig", dtype=str)
        sumber.append((d["file"], d["bidang"], df))
    for nama, _, df in sumber:
        catat("1-baca", f"{nama}: {len(df)} baris, {df.shape[1]} kolom")

    bagian, raws = [], []
    for urut, (nama, bidang, df) in enumerate(sumber):
        r = df.copy()
        r.insert(0, "file_sumber", nama)
        raws.append(r)
        c = df.copy()
        c.columns = [snake(x) for x in c.columns]
        c = c.apply(lambda s: s.map(clean_text))
        c["file_sumber"] = nama
        c["nomor_baris"] = range(1, len(c) + 1)
        c["urut_sumber"] = urut
        bagian.append(c)
    # kolom gabungan: urutan kemunculan, 'eid' dipindah ke akhir seperti contoh prototype
    kolom = []
    for c in bagian:
        kolom += [x for x in c.columns if x not in kolom and x not in ("file_sumber", "nomor_baris", "urut_sumber")]
    kolom = [x for x in kolom if x != "eid"] + ["eid"]
    R = pd.concat([c.reindex(columns=kolom + ["file_sumber", "nomor_baris", "urut_sumber"]) for c in bagian], ignore_index=True)
    for wajib in ("eid", "doi", "title", "authors", "author_full_names", "author_s_id", "year", "source_title", "cited_by",
                  "affiliations", "author_keywords", "index_keywords", "funding_details", "funding_texts", "references", "abstract"):
        if wajib not in R:
            R[wajib] = None
            catat("1-baca", f"PERHATIAN: kolom '{wajib}' tidak ada di semua file -> dikosongkan")
    R["citation_count"] = pd.to_numeric(R["cited_by"], errors="coerce").fillna(0).astype(int)
    R["src_key"] = R["file_sumber"] + ":" + R["nomor_baris"].astype(str)
    R["doi_norm"] = R["doi"].map(norm_doi)
    R["title_key"] = R["title"].map(judul_key)
    R["kunci_eid"] = R["eid"].where(R["eid"].notna(), R["src_key"])
    R["authors_list"] = [author_list(r) for r in R.to_dict("records")]
    R["terisi"] = R[kolom].notna().sum(axis=1)

    # cadangan data mentah (tidak menimpa; versi baru bila isi berubah)
    manifest = []
    for d in FILES:
        p = os.path.join(BASE_DIR, d["file"])
        with open(p, "rb") as f:
            sha = hashlib.sha1(f.read()).hexdigest()
        manifest.append({"file": d["file"], "sha1": sha, "ukuran_byte": os.path.getsize(p), "waktu_cek": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
        if SIMPAN_HASIL:
            os.makedirs(BACKUP_DIR, exist_ok=True)
            tujuan = os.path.join(BACKUP_DIR, f"{sha[:8]}__{d['file']}")
            if not os.path.exists(tujuan):
                shutil.copy2(p, tujuan)
    if SIMPAN_HASIL:
        tulis_csv(pd.DataFrame(manifest), os.path.join(BACKUP_DIR, "manifest_sha1.csv"))
    return R, kolom, pd.concat(raws, ignore_index=True), prototype, sumber

R, KOLOM, RAW_SEMUA, PROTOTYPE, SUMBER = tahap1_baca()
print("Total baris semua sumber:", len(R))

[1-baca] AGRICULTURE.csv: 4150 baris, 45 kolom
[1-baca] Computer Science Mathematics Decision Sciences.csv: 1200 baris, 46 kolom
[1-baca] Energy.csv: 1142 baris, 45 kolom
[1-baca] Ilmu Hayati.csv: 1785 baris, 45 kolom
[1-baca] Ilmu Lingkungan.csv: 3191 baris, 45 kolom
[1-baca] Ilmu Material.csv: 1690 baris, 45 kolom
[1-baca] Medicine.csv: 1257 baris, 45 kolom
[1-baca] Physics and Astronomy Earth and Planetary Sciences.csv: 149 baris, 45 kolom
[1-baca] Veteriner.csv: 575 baris, 45 kolom
Total baris semua sumber: 15139


## Tahap 2 — Pemeriksaan struktur & kualitas data
Kelengkapan kolom penting (DOI, Title, Authors, EID), format DOI, dan EID kembar dalam satu file. Hasilnya masuk laporan.

In [4]:
def tahap2_periksa(R):
    baris = []
    for nama, g in R.groupby("file_sumber", sort=False):
        doi_ada = g["doi_norm"].notna()
        baris.append({
            "file": nama, "baris_awal": len(g),
            "doi_kosong": int((~doi_ada).sum()),
            "doi_format_salah": int((doi_ada & ~g["doi_norm"].fillna("").map(lambda x: bool(DOI_RE.match(x)))).sum()),
            "title_kosong": int(g["title"].isna().sum()), "authors_kosong": int(g["authors"].isna().sum()),
            "eid_kosong": int(g["eid"].isna().sum()),
            "eid_kembar": int(g["eid"].dropna().duplicated().sum()),
            "doi_kembar": int(g["doi_norm"].dropna().duplicated().sum()),
            "judul_kembar": int((g["title_key"] != "").pipe(lambda m: g.loc[m, "title_key"].duplicated().sum())),
        })
    kualitas = pd.DataFrame(baris)
    for r in baris:
        catat("2-periksa", f"{r['file']}: {r['baris_awal']} baris | DOI kosong {r['doi_kosong']} | DOI format salah {r['doi_format_salah']} | "
                           f"EID kembar {r['eid_kembar']} | DOI kembar {r['doi_kembar']} | judul kembar {r['judul_kembar']}")
    return kualitas

KUALITAS = tahap2_periksa(R)
display(KUALITAS) if "display" in globals() else print(KUALITAS.to_string())

[2-periksa] AGRICULTURE.csv: 4150 baris | DOI kosong 190 | DOI format salah 0 | EID kembar 0 | DOI kembar 1 | judul kembar 2
[2-periksa] Computer Science Mathematics Decision Sciences.csv: 1200 baris | DOI kosong 22 | DOI format salah 0 | EID kembar 0 | DOI kembar 0 | judul kembar 0
[2-periksa] Energy.csv: 1142 baris | DOI kosong 9 | DOI format salah 0 | EID kembar 0 | DOI kembar 0 | judul kembar 0
[2-periksa] Ilmu Hayati.csv: 1785 baris | DOI kosong 9 | DOI format salah 0 | EID kembar 0 | DOI kembar 0 | judul kembar 0
[2-periksa] Ilmu Lingkungan.csv: 3191 baris | DOI kosong 172 | DOI format salah 0 | EID kembar 0 | DOI kembar 1 | judul kembar 1
[2-periksa] Ilmu Material.csv: 1690 baris | DOI kosong 39 | DOI format salah 0 | EID kembar 0 | DOI kembar 0 | judul kembar 1
[2-periksa] Medicine.csv: 1257 baris | DOI kosong 50 | DOI format salah 0 | EID kembar 0 | DOI kembar 0 | judul kembar 0
[2-periksa] Physics and Astronomy Earth and Planetary Sciences.csv: 149 baris | DOI kosong 0 | DOI 

## Tahap 3 & 6 — Kandidat duplikat (dalam file / antar-file)
Kandidat dicari lewat indeks (EID, DOI, judul) sehingga cepat untuk ribuan baris. Satu pasangan baris = satu kandidat dengan
daftar bukti: `doi` sama, `judul` sama (+ rasio author). EID tidak dipakai.

In [5]:
def tahap_kandidat(R, lingkup):
    """lingkup: 'dalam_file' (kedua baris dari file sama) atau 'antar_file' (file beda). Return daftar kandidat."""
    sama_file = lambda i, j: R.at[i, "file_sumber"] == R.at[j, "file_sumber"]
    ok = (lambda i, j: sama_file(i, j)) if lingkup == "dalam_file" else (lambda i, j: not sama_file(i, j))
    pasangan = {}
    def tambah(i, j, bukti):
        if i == j or not ok(i, j):
            return
        k = (min(i, j), max(i, j))
        pasangan.setdefault(k, set()).add(bukti)
    for kolom, bukti in (("doi_norm", "doi"), ("title_key", "judul")):
        s = R[kolom]
        s = s[s.notna() & (s != "")]
        for _, idx in s.groupby(s).groups.items():
            idx = list(idx)
            if len(idx) < 2:
                continue
            for a in range(len(idx)):
                for b in range(a + 1, len(idx)):
                    tambah(idx[a], idx[b], bukti)
    out = []
    for (i, j), bukti in pasangan.items():
        rasio = rasio_author(R.at[i, "authors_list"], R.at[j, "authors_list"])
        out.append({"i": i, "j": j, "bukti": bukti, "rasio": rasio, "lingkup": lingkup})
    return out

## Tahap 4 & 7 — Validasi DOI (terpisah, sebelum merge)
Satu DOI dianggap **valid** hanya jika lolos tiga lapis: (1) format `10.xxxx/...`, (2) dapat diakses lewat resolver DOI
(metadata diminta dari `doi.org`, cadangan Crossref), (3) metadata **cocok** dengan baris di CSV (judul, tahun, nama belakang author pertama).
Berhasil dibuka saja **tidak cukup**. Status: `valid_cocok` · `valid_terdaftar` (ada & bisa dibuka, tanpa metadata) · `valid_tidak_cocok` ·
`tidak_dapat_diakses` (server gagal/membatasi, sudah dicoba ulang) · `tidak_ditemukan` · `format_salah` · `kosong`.
Semua DOI dicek otomatis; permintaan yang gagal dicoba ulang dengan jeda makin lama dan hasilnya disimpan bertahap.
Hasil permintaan disimpan di cache (`laporan/cache_validasi_doi.csv`) sehingga Run berikutnya tidak mengulang permintaan.

In [6]:
PENGAMBIL_DOI = None     # (untuk uji) fungsi pengganti: f(doi) -> dict(status, title, year, authors). None = pakai internet

def _csl_ke_meta(j):
    judul = j.get("title")
    judul = judul[0] if isinstance(judul, list) and judul else judul
    tahun = None
    for k in ("issued", "published-print", "published-online", "published"):
        dp = (j.get(k) or {}).get("date-parts")
        if dp and dp[0] and dp[0][0]:
            tahun = str(dp[0][0]); break
    fam = [a.get("family") or a.get("name") or "" for a in (j.get("author") or [])]
    if not judul:                                   # DOI terdaftar tetapi tanpa metadata judul
        return {"status": "terdaftar"}
    return {"status": "ok", "title": judul, "year": tahun or "", "authors": "; ".join(fam)}

_TLS = threading.local()
def _sesi():
    import requests
    if not hasattr(_TLS, "s"):
        _TLS.s = requests.Session()
        _TLS.s.headers["User-Agent"] = "normalisasi-scopus/1.0" + (f" (mailto:{DOI_EMAIL})" if DOI_EMAIL else "")
    return _TLS.s

def _get(url, **kw):
    """GET dengan percobaan ulang (429 / 5xx / timeout -> jeda makin lama). Return response, atau None bila tetap gagal."""
    for k in range(DOI_COBA_ULANG):
        try:
            res = _sesi().get(url, timeout=DOI_TIMEOUT, **kw)
        except Exception:
            time.sleep(2 ** k); continue
        if res.status_code == 429 or res.status_code >= 500:
            ra = res.headers.get("Retry-After", "")
            time.sleep(min(float(ra), 60) if ra.replace(".", "", 1).isdigit() else 2 ** k * 2); continue
        return res
    return None

def ambil_doi_online(doi):
    """Minta metadata DOI. status: ok | terdaftar (DOI ada, tanpa metadata) | tidak_ditemukan | tidak_dapat_diakses"""
    d = quote(doi, safe="/")
    a = _get(f"https://doi.org/{d}", allow_redirects=True, headers={"Accept": "application/vnd.citationstyles.csl+json"})
    if a is not None and a.status_code == 200 and "json" in a.headers.get("content-type", ""):
        try: return _csl_ke_meta(a.json())
        except Exception: pass
    c = _get(f"https://api.crossref.org/works/{d}")                      # DOI Crossref yang tidak memberi metadata lewat doi.org
    if c is not None and c.status_code == 200:
        try: return _csl_ke_meta(c.json()["message"])
        except Exception: pass
    h = _get(f"https://doi.org/api/handles/{d}")                         # DOI non-Crossref: cukup cek apakah terdaftar
    if h is not None and h.status_code in (200, 404):
        try: kode = h.json().get("responseCode")
        except Exception: kode = None
        if kode == 1: return {"status": "terdaftar"}
        if kode == 100 or h.status_code == 404: return {"status": "tidak_ditemukan"}
    if h is None and a is not None and a.status_code == 404 and c is not None and c.status_code == 404:
        return {"status": "tidak_ditemukan"}
    return {"status": "tidak_dapat_diakses"}

# cache hasil permintaan DOI (disimpan antar-Run)
DOI_CACHE = {}
if os.path.exists(FILE_CACHE_DOI):
    for _r in pd.read_csv(FILE_CACHE_DOI, dtype=str, encoding="utf-8-sig").fillna("").to_dict("records"):
        DOI_CACHE[_r["doi"]] = {"status": _r["status"], "title": _r["title"], "year": _r["year"], "authors": _r["authors"], "waktu": _r["waktu"]}

def simpan_cache():
    if not SIMPAN_HASIL:
        return
    os.makedirs(LAPORAN_DIR, exist_ok=True)
    tulis_csv(pd.DataFrame([{"doi": d, **{k: v for k, v in c.items() if k != "dicoba_run_ini"}} for d, c in DOI_CACHE.items()],
                           columns=["doi", "status", "title", "year", "authors", "waktu"]), FILE_CACHE_DOI)

def validasi_doi_batch(daftar_doi, tahap):
    """Ambil metadata untuk DOI yang belum ada di cache (paralel, disimpan tiap 300 DOI -> aman bila terputus).
    'tidak_dapat_diakses' & 'tidak_ditemukan' lama dicoba ulang 1x per Run."""
    if VALIDASI_DOI == "mati":
        return
    ulang = ("tidak_dapat_diakses", "tidak_ditemukan")
    baru = [d for d in dict.fromkeys(daftar_doi)
            if d and DOI_RE.match(d) and (d not in DOI_CACHE or DOI_CACHE[d]["status"] in ulang and not DOI_CACHE[d].get("dicoba_run_ini"))]
    if not baru:
        catat(tahap, "validasi DOI: semua DOI sudah ada di cache")
        return
    catat(tahap, f"validasi DOI: meminta metadata {len(baru)} DOI ...")
    fetch = PENGAMBIL_DOI or ambil_doi_online
    for a in range(0, len(baru), 300):
        potong = baru[a:a + 300]
        with ThreadPoolExecutor(max_workers=DOI_PARALEL) as ex:
            hasil = list(ex.map(fetch, potong))
        for d, h in zip(potong, hasil):
            DOI_CACHE[d] = {"status": h.get("status", "tidak_dapat_diakses"), "title": h.get("title", ""), "year": h.get("year", ""),
                            "authors": h.get("authors", ""), "waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "dicoba_run_ini": True}
        simpan_cache()
        print(f"  ... {min(a + 300, len(baru))}/{len(baru)} DOI diperiksa")
    st = pd.Series([DOI_CACHE[d]["status"] for d in baru]).value_counts().to_dict()
    catat(tahap, f"validasi DOI: hasil {st}")

def _lipat(t):
    """Teks -> huruf kecil tanpa aksen, tanpa tag HTML/MathML."""
    t = html.unescape(re.sub(r"<[^>]+>", " ", str(t or "")))
    t = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", t))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).casefold()

def skor_judul(a, b):
    """Kemiripan 2 judul (0-1): kunci huruf+angka, atau tumpang-tindih kata (tahan terhadap rumus/simbol/tag HTML)."""
    a, b = _lipat(a), _lipat(b)
    ka, kb = re.sub(r"[\W_]+", "", a), re.sub(r"[\W_]+", "", b)
    if not ka or not kb:
        return 0.0
    if ka == kb or ((ka.startswith(kb) or kb.startswith(ka)) and min(len(ka), len(kb)) >= 15):
        return 1.0
    r1 = SequenceMatcher(None, ka, kb).ratio()
    wa, wb = set(re.findall(r"[a-z]{3,}", a)), set(re.findall(r"[a-z]{3,}", b))
    r2 = len(wa & wb) / min(len(wa), len(wb)) if min(len(wa), len(wb)) >= 4 else 0.0
    return max(r1, r2)

def nilai_cocok(title, year, authors, c):
    """Bandingkan judul/tahun/author di CSV dengan metadata DOI (c). Return dict(skor, tahun_ok, author_ok, cocok).
    Cocok bila: (judul >= AMBANG_JUDUL_DOI dan panjang) ATAU (judul >= AMBANG_JUDUL_LONGGAR dan tahun +-1 dan ada author cocok)."""
    skor = skor_judul(title, c["title"])
    th_csv, th_doi = clean_text(year), c["year"]
    tahun_ok = abs(int(th_csv) - int(th_doi)) <= 1 if (th_csv or "").isdigit() and (th_doi or "").isdigit() else True
    nama_csv = [re.sub(r"[\W_]+", "", _lipat(a.split()[0])) for a in split_semicolon(authors)[:3] if a.split()]
    meta_authors = re.sub(r"[\W_]+", "", _lipat(c["authors"]))
    author_ok = (not nama_csv) or (not meta_authors) or any(n and n in meta_authors for n in nama_csv)
    panjang = len(re.sub(r"[\W_]+", "", _lipat(title))) >= 25
    cocok = (skor >= AMBANG_JUDUL_DOI and (panjang or (tahun_ok and author_ok))) or (skor >= AMBANG_JUDUL_LONGGAR and tahun_ok and author_ok)
    return {"skor": skor, "tahun_ok": tahun_ok, "author_ok": author_ok, "cocok": bool(cocok)}

def cocok_meta(title, year, authors, c):
    return nilai_cocok(title, year, authors, c)["cocok"]

def status_doi(i):
    """Status DOI baris R[i] (cocok tidaknya dihitung terhadap judul/tahun/author baris itu)."""
    doi = R.at[i, "doi_norm"]
    if not ada(doi):
        return "kosong"
    if not DOI_RE.match(doi):
        return "format_salah"
    if VALIDASI_DOI == "mati":
        return "belum_divalidasi"
    c = DOI_CACHE.get(doi)
    if c is None:
        return "belum_divalidasi"           # tidak termasuk kandidat duplikat (mode "kandidat")
    if c["status"] == "tidak_dapat_diakses":
        return "tidak_dapat_diakses"
    if c["status"] == "tidak_ditemukan":
        return "tidak_ditemukan"
    if c["status"] == "terdaftar":
        return "valid_terdaftar"            # DOI ada & bisa dibuka, tetapi tidak ada metadata untuk dicocokkan
    # metadata terbaca -> cocokkan dengan baris CSV
    return "valid_cocok" if cocok_meta(R.at[i, "title"], R.at[i, "year"], R.at[i, "authors"], c) else "valid_tidak_cocok"

## Tahap 5 & 8 — Keputusan dan merge (otomatis)
Dua record digabung hanya jika **ketiga syarat terpenuhi, diperiksa berurutan** (EID tidak dipakai):
1. **DOI** — salah satu: DOI sama · kedua DOI kosong · salah satu kosong & DOI lainnya terverifikasi (bisa dibuka) · DOI berbeda tetapi
   **tepat satu** yang terverifikasi (digabung ke yang terverifikasi). Dua DOI berbeda yang sama-sama valid = dua karya berbeda
   (mis. preprint vs artikel) → tidak digabung.
2. **Judul** — sama setelah normalisasi (huruf besar/kecil, tanda baca, spasi diabaikan).
3. **Author** — minimal `AMBANG_AUTHOR_SAMA` (80%) sama.
Gagal di salah satu → **tidak digabung** (alasan dicatat di log). Bila kedua DOI kosong, DOI hasil merge diisi EID.

In [7]:
# (opsional) paksa gabung lewat PAKSA_GABUNG di pengaturan
FORCE = {"|".join(sorted(p)) for p in PAKSA_GABUNG}
VALID_OK = ("valid_cocok", "valid_terdaftar")

class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def cari(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def gabung(self, a, b):
        a, b = self.cari(a), self.cari(b)
        if a != b: self.p[max(a, b)] = min(a, b)        # akar = baris paling awal (urutan sumber, lalu baris)

def putuskan(e):
    """Return (keputusan, alasan): keputusan = 'auto' (gabung) | 'tolak' (tidak digabung). Syarat: DOI -> judul -> author."""
    i, j, bukti, rasio = e["i"], e["j"], e["bukti"], e["rasio"]
    kunci = "|".join(sorted((R.at[i, "kunci_eid"], R.at[j, "kunci_eid"])))
    e["kunci"] = kunci
    if kunci in FORCE:
        return "auto", "dipaksa gabung (PAKSA_GABUNG)"
    si, sj = status_doi(i), status_doi(j)
    e["status_doi_i"], e["status_doi_j"] = si, sj
    di, dj = R.at[i, "doi_norm"], R.at[j, "doi_norm"]
    ada_i, ada_j = ada(di), ada(dj)
    oi, oj = si in VALID_OK, sj in VALID_OK
    mati = not (WAJIB_DOI_VALID and VALIDASI_DOI != "mati")
    # --- syarat 1: DOI
    if not ada_i and not ada_j:
        s1 = "kedua DOI kosong"
    elif ada_i and ada_j and di == dj:
        s1 = "DOI sama"
    elif ada_i != ada_j:
        if mati or (oi if ada_i else oj):
            s1 = f"satu DOI kosong, DOI lainnya {'dipakai' if mati else 'terverifikasi'}"
        else:
            return "tolak", f"tidak digabung [DOI]: satu DOI kosong dan DOI lainnya belum terverifikasi ({si} / {sj})"
    else:           # dua DOI berbeda
        if mati:
            return "tolak", "tidak digabung [DOI]: DOI berbeda"
        if oi and oj:
            return "tolak", f"tidak digabung [DOI]: dua DOI berbeda sama-sama valid ({si} / {sj}) -> dua karya berbeda"
        if oi or oj:
            s1 = f"DOI berbeda, digabung ke DOI yang terverifikasi ({si} / {sj})"
        else:
            return "tolak", f"tidak digabung [DOI]: DOI berbeda dan tidak ada yang terverifikasi ({si} / {sj})"
    # --- syarat 2: judul
    if "judul" in bukti:
        s2 = "judul sama"
    else:
        sim = SequenceMatcher(None, judul_key(R.at[i, "title"]), judul_key(R.at[j, "title"])).ratio()
        if s1 == "DOI sama" and sim >= AMBANG_JUDUL_SAMA:
            s2 = f"judul mirip ({sim:.0%})"
        else:
            return "tolak", f"tidak digabung [judul]: judul berbeda ({s1})"
    # --- syarat 3: author
    if rasio < AMBANG_AUTHOR_SAMA:
        return "tolak", f"tidak digabung [author]: author hanya {rasio:.0%} sama ({s1}; {s2})"
    return "auto", f"{s1}; {s2}; author {rasio:.0%} sama"

UF = UnionFind(len(R))
KEPUTUSAN_LOG, STAT_TAHAP = [], {}

def proses_lingkup(lingkup, tahap_kand, tahap_doi, tahap_merge):
    kand = tahap_kandidat(R, lingkup)
    catat(tahap_kand, f"{len(kand)} pasangan kandidat duplikat ({lingkup.replace('_', ' ')})")
    # validasi DOI: kandidat (default) atau semua DOI
    if VALIDASI_DOI == "semua":
        dois = list(R["doi_norm"].dropna())
    else:
        dois = [R.at[k, "doi_norm"] for e in kand for k in (e["i"], e["j"]) if ada(R.at[k, "doi_norm"])]
    validasi_doi_batch(dois, tahap_doi)
    n_auto = n_tolak = 0
    for e in kand:
        k, alasan = putuskan(e)
        e["keputusan"], e["alasan"] = k, alasan
        if k == "auto":
            UF.gabung(e["i"], e["j"]); n_auto += 1
        else:
            n_tolak += 1
        KEPUTUSAN_LOG.append(e)
    STAT_TAHAP[lingkup] = {"kandidat": len(kand), "digabung": n_auto, "tidak_digabung": n_tolak}
    catat(tahap_merge, f"{n_auto} pasangan digabung | {n_tolak} tidak digabung (bukti kurang / publikasi berbeda)")

# Tahap 3-5: dalam file
proses_lingkup("dalam_file", "3-dup-dalam-file", "4-validasi-doi", "5-merge-dalam-file")
R["klaster_dalam_file"] = [UF.cari(i) for i in range(len(R))]
# Tahap 6-8: antar-file
proses_lingkup("antar_file", "6-dup-antar-file", "7-validasi-doi", "8-merge-antar-file")
R["klaster"] = [UF.cari(i) for i in range(len(R))]
print("Publikasi unik setelah merge:", R["klaster"].nunique(), "dari", len(R), "baris")

[3-dup-dalam-file] 6 pasangan kandidat duplikat (dalam file)
[4-validasi-doi] validasi DOI: meminta metadata 9922 DOI ...
  ... 300/9922 DOI diperiksa
  ... 600/9922 DOI diperiksa
  ... 900/9922 DOI diperiksa
  ... 1200/9922 DOI diperiksa
  ... 1500/9922 DOI diperiksa
  ... 1800/9922 DOI diperiksa
  ... 2100/9922 DOI diperiksa
  ... 2400/9922 DOI diperiksa
  ... 2700/9922 DOI diperiksa
  ... 3000/9922 DOI diperiksa
  ... 3300/9922 DOI diperiksa
  ... 3600/9922 DOI diperiksa
  ... 3900/9922 DOI diperiksa
  ... 4200/9922 DOI diperiksa
  ... 4500/9922 DOI diperiksa
  ... 4800/9922 DOI diperiksa
  ... 5100/9922 DOI diperiksa
  ... 5400/9922 DOI diperiksa
  ... 5700/9922 DOI diperiksa
  ... 6000/9922 DOI diperiksa
  ... 6300/9922 DOI diperiksa
  ... 6600/9922 DOI diperiksa
  ... 6900/9922 DOI diperiksa
  ... 7200/9922 DOI diperiksa
  ... 7500/9922 DOI diperiksa
  ... 7800/9922 DOI diperiksa
  ... 8100/9922 DOI diperiksa
  ... 8400/9922 DOI diperiksa
  ... 8700/9922 DOI diperiksa
  ... 9000/

## Tahap 9 — `publication_id` berurutan tanpa celah
ID dibuat **setelah** semua duplikat selesai: satu klaster = satu `publication_id`, berurutan menurut kemunculan pertama
(prototype lebih dulu → file sesuai urutan daftar → urutan baris). Prototype mempertahankan ID aslinya (PUB000001-PUB000020).
Karena semua tabel relasi dibangun dari ID akhir ini, foreign key otomatis konsisten.

In [8]:
def tahap9_id(R):
    # ID prototype tetap
    id_proto = {}
    if PROTOTYPE:
        cl = PROTOTYPE["CLEAN"]
        id_proto = dict(zip(cl["eid"], cl["publication_id"]))
    urut = list(dict.fromkeys(R.sort_values(["urut_sumber", "nomor_baris"])["klaster"]))     # klaster menurut kemunculan pertama
    anggota = R.groupby("klaster").groups
    pid, nomor = {}, max([int(re.sub(r"\D", "", v) or 0) for v in id_proto.values()] + [0])
    for kl in urut:                                            # klaster yang memuat baris prototype memakai ID prototype
        for idx in anggota[kl]:
            if R.at[idx, "file_sumber"] == FILE_ACUAN and R.at[idx, "eid"] in id_proto:
                pid[kl] = id_proto[R.at[idx, "eid"]]; break
    for kl in urut:
        if kl not in pid:
            nomor += 1
            pid[kl] = f"PUB{nomor:06d}"
    R["publication_id"] = R["klaster"].map(pid)
    return R

R = tahap9_id(R)
_nomor = sorted(int(x[3:]) for x in R["publication_id"].unique())
assert _nomor == list(range(1, len(_nomor) + 1)), "ID publikasi tidak rapat (ada celah) -> periksa data acuan"
catat("9-id", f"{len(_nomor)} publication_id unik: PUB000001 s/d PUB{_nomor[-1]:06d} (tanpa celah)")

# perubahan ID dibanding Run sebelumnya (log_perubahan_id.csv)
PERUBAHAN_ID = pd.DataFrame(columns=["eid", "file_sumber", "id_lama", "id_baru"])
if os.path.exists(FILE_PEMETAAN):
    _lama = pd.read_csv(FILE_PEMETAAN, dtype=str, encoding="utf-8-sig")[["file_sumber", "kunci_eid", "publication_id"]].rename(columns={"publication_id": "id_lama"})
    _m = R[["file_sumber", "kunci_eid", "publication_id"]].merge(_lama, on=["file_sumber", "kunci_eid"], how="inner")
    PERUBAHAN_ID = (_m[_m["id_lama"] != _m["publication_id"]].rename(columns={"publication_id": "id_baru", "kunci_eid": "eid"})
                     [["eid", "file_sumber", "id_lama", "id_baru"]].reset_index(drop=True))
    catat("9-id", f"perubahan ID dibanding Run sebelumnya: {len(PERUBAHAN_ID)} baris")

[9-id] 10227 publication_id unik: PUB000001 s/d PUB010227 (tanpa celah)


## Merge — satu record utama per publikasi (tanpa menghilangkan informasi)
Record utama = baris dengan kolom terisi terbanyak (seri: paling awal). Kolom kosong diisi dari baris lain; keyword & funding
digabung (tanpa duplikat); `citation_count` = yang terbesar; DOI = DOI yang tervalidasi bila ada. Semua baris asli tetap ada di
sheet **CLEAN** (dengan `publication_id` + `file_sumber`) dan di `log_merge.csv` (EID/DOI semua sumber).

In [9]:
KOLOM_UNION = ["author_keywords", "index_keywords", "funding_details"]

# --- Koreksi DOI otomatis: DOI yang tercatat harus cocok dengan publikasinya
DOI_SALAH = ("valid_tidak_cocok", "tidak_ditemukan", "format_salah")
PENCARI_DOI = None     # (untuk uji) fungsi pengganti: f(judul, tahun, nama_belakang) -> dict(status, doi). None = Crossref
FILE_CACHE_KOREKSI = os.path.join(LAPORAN_DIR, "cache_koreksi_doi_v2.csv")
KOREKSI_CACHE = {}
if os.path.exists(FILE_CACHE_KOREKSI):
    for _r in pd.read_csv(FILE_CACHE_KOREKSI, dtype=str, encoding="utf-8-sig").fillna("").to_dict("records"):
        KOREKSI_CACHE[_r["kunci"]] = {"status": _r["status"], "doi": _r["doi"]}

def cari_doi_online(judul, tahun, nama_belakang):
    """Cari DOI yang benar lewat Crossref (judul + author). Hanya DOI yang metadatanya cocok (judul/tahun/author) yang diterima."""
    res = _get("https://api.crossref.org/works", params={"query.bibliographic": f"{judul} {nama_belakang}".strip(), "rows": 5,
                                                        "select": "DOI,title,issued,author"})
    if res is None or res.status_code != 200:
        return {"status": "tidak_dapat_diakses"}
    try:
        items = res.json()["message"]["items"]
    except Exception:
        return {"status": "tidak_dapat_diakses"}
    for it in items:
        m = _csl_ke_meta({"title": it.get("title"), "issued": it.get("issued"), "author": it.get("author")})
        if m["status"] == "ok" and cocok_meta(judul, tahun, nama_belakang, m):
            return {"status": "ok", "doi": it["DOI"].lower()}
    return {"status": "tidak_ditemukan"}

def simpan_cache_koreksi():
    if SIMPAN_HASIL:
        os.makedirs(LAPORAN_DIR, exist_ok=True)
        tulis_csv(pd.DataFrame([{"kunci": k, **v} for k, v in KOREKSI_CACHE.items()], columns=["kunci", "status", "doi"]), FILE_CACHE_KOREKSI)

def koreksi_doi_batch(pub_rows, perlu):
    """DOI bermasalah -> (1) cari DOI benar di Crossref; (2) bila tidak ada: DOI yang TIDAK ditemukan / formatnya salah diganti EID
    (DOI asli disimpan di kolom doi_asli), sedangkan DOI yang bisa dibuka tetapi metadatanya beda tetap dipertahankan (diberi tanda).
    Bila server tidak dapat dihubungi, DOI dibiarkan dan dicoba lagi di Run berikutnya."""
    if not perlu or not KOREKSI_DOI or VALIDASI_DOI == "mati":
        return
    def kunci_q(k):
        pertama = split_semicolon(R.at[k, "authors"])
        return (R.at[k, "title"] or "", clean_text(R.at[k, "year"]) or "", judul_key(pertama[0].split()[0]) if pertama else "")
    kunci_str = lambda q: judul_key(q[0]) + "|" + q[1]
    baru = {}
    for _, k in perlu:
        q = kunci_q(k); ks = kunci_str(q)
        if ks not in KOREKSI_CACHE or KOREKSI_CACHE[ks]["status"] == "tidak_dapat_diakses":
            baru[ks] = q
    if baru:
        catat("8-koreksi-doi", f"{len(perlu)} publikasi dengan DOI salah -> mencari DOI yang benar ({len(baru)} pencarian) ...")
        fetch = PENCARI_DOI or cari_doi_online
        daftar = list(baru.items())
        with ThreadPoolExecutor(max_workers=DOI_PARALEL) as ex:
            for (ks, _), h in zip(daftar, ex.map(lambda x: fetch(*x[1]), daftar)):
                KOREKSI_CACHE[ks] = {"status": h.get("status", "tidak_dapat_diakses"), "doi": h.get("doi", "") or ""}
        simpan_cache_koreksi()
    n_ok = n_eid = n_tunda = 0
    for pos, k in perlu:
        rec = pub_rows[pos]
        h = KOREKSI_CACHE[kunci_str(kunci_q(k))]
        if h["status"] == "ok" and h["doi"]:
            rec["doi_asli"], rec["doi"], rec["doi_status"] = rec["doi"], h["doi"], "dikoreksi_otomatis"; n_ok += 1
        elif h["status"] == "tidak_ditemukan" and rec["doi_status"] == "valid_tidak_cocok":
            n_tunda += 0                  # DOI bisa dibuka (terdaftar) tetapi metadata beda & tidak ada DOI lain yang cocok -> DOI asli dipertahankan, diberi tanda
        elif h["status"] == "tidak_ditemukan":
            rec["doi_asli"], rec["doi"], rec["doi_status"] = rec["doi"], rec["eid"], rec["doi_status"] + "_diganti_eid"; n_eid += 1
        else:
            n_tunda += 1
    n_tetap = sum(1 for pos, _ in perlu if pub_rows[pos]["doi_status"] == "valid_tidak_cocok" and KOREKSI_CACHE[kunci_str(kunci_q(_))]["status"] == "tidak_ditemukan")
    catat("8-koreksi-doi", f"{n_ok} DOI dikoreksi otomatis | {n_eid} diganti EID (DOI tidak ditemukan & DOI benar tidak ada) | "
                           f"{n_tetap} DOI bisa dibuka tetapi metadata beda -> DOI asli dipertahankan | {n_tunda - n_tetap if n_tunda >= n_tetap else 0} ditunda (server tidak dapat dihubungi)")

def tahap_merge_record(R, kolom):
    anggota = R.groupby("publication_id", sort=False).groups
    pub_rows, log = [], []
    alasan_node = {}
    for e in KEPUTUSAN_LOG:
        if e["keputusan"] == "auto":
            alasan_node.setdefault(e["i"], set()).add(e["alasan"]); alasan_node.setdefault(e["j"], set()).add(e["alasan"])
    perlu = []                                    # (posisi di pub_rows, baris DOI) publikasi yang DOI-nya salah -> dikoreksi otomatis
    for pid, idx in anggota.items():
        idx = sorted(idx, key=lambda k: (R.at[k, "urut_sumber"], R.at[k, "nomor_baris"]))
        if len(idx) == 1:
            utama = idx[0]
        else:       # record utama = baris yang DOI-nya bisa diakses/cocok; seri -> terlengkap (atau paling atas)
            utama = max(idx, key=lambda k: (status_doi(k) == "valid_cocok", status_doi(k) in VALID_OK,
                                            R.at[k, "terisi"] if PILIH_BARIS_UTAMA == "terlengkap" else 0, -idx.index(k)))
        rec = {c: R.at[utama, c] for c in kolom}
        rec["citation_count"] = int(max(R.at[k, "citation_count"] for k in idx))
        if len(idx) > 1:
            lain = [k for k in idx if k != utama]
            for c in kolom:
                if not ada(rec[c]) and not (c == "citation_count"):
                    for k in lain:
                        v = R.at[k, c]
                        if ada(v):
                            rec[c] = v; break
            for c in KOLOM_UNION:
                bagian, lihat = [], set()
                for k in [utama] + lain:
                    for p in split_semicolon(R.at[k, c]):
                        if p.lower() not in lihat:
                            lihat.add(p.lower()); bagian.append(p)
                rec[c] = "; ".join(bagian) if bagian else None
            valid = sorted([k for k in idx if ada(R.at[k, "doi"]) and status_doi(k) in VALID_OK], key=lambda k: status_doi(k) != "valid_cocok")
            if valid:
                rec["doi"] = R.at[valid[0], "doi"]
            elif not ada(rec["doi"]):                       # DOI utama kosong -> pakai DOI baris lain bila ada
                rec["doi"] = next((R.at[k, "doi"] for k in idx if ada(R.at[k, "doi"])), None)
            alasan = sorted(set().union(*[alasan_node.get(k, set()) for k in idx]))
            log.append({"publication_id": pid, "jumlah_baris": len(idx),
                        "file_sumber": " | ".join(dict.fromkeys(R.at[k, "file_sumber"] for k in idx)),
                        "eid": " | ".join(dict.fromkeys(str(R.at[k, "eid"]) for k in idx if ada(R.at[k, "eid"]))),
                        "doi": " | ".join(dict.fromkeys(str(R.at[k, "doi"]) for k in idx if ada(R.at[k, "doi"]))),
                        "doi_dipakai": rec.get("doi"), "baris_utama": R.at[utama, "src_key"],
                        "alasan_merge": " ; ".join(alasan)})
        rec["publication_id"] = pid
        _k = next((k for k in idx if R.at[k, "doi"] == rec.get("doi")), idx[0]) if ada(rec.get("doi")) else idx[0]
        if ada(rec.get("doi")) and ada(R.at[_k, "doi_norm"]) and R.at[_k, "doi_norm"] != _doi_dasar(rec["doi"]):
            rec["doi_dibersihkan_dari"] = rec["doi"]; rec["doi"] = R.at[_k, "doi_norm"]      # mis. prefix terulang / titik di ujung dibuang
        _st = status_doi(_k) if ada(rec.get("doi")) else "kosong"
        rec["doi_status"] = _st
        rec["doi_asli"] = rec.pop("doi_dibersihkan_dari", None)
        if _st in DOI_SALAH:
            perlu.append((len(pub_rows), _k))
        pub_rows.append(rec)
    koreksi_doi_batch(pub_rows, perlu)
    pub = pd.DataFrame(pub_rows)
    return pub, pd.DataFrame(log, columns=["publication_id", "jumlah_baris", "file_sumber", "eid", "doi", "doi_dipakai", "baris_utama", "alasan_merge"])

PUB, LOG_MERGE = tahap_merge_record(R, KOLOM)
catat("8-merge", f"{len(LOG_MERGE)} publikasi hasil merge (gabungan >1 baris) | total {len(PUB)} publikasi unik")

[8-koreksi-doi] 131 publikasi dengan DOI salah -> mencari DOI yang benar (131 pencarian) ...
[8-koreksi-doi] 44 DOI dikoreksi otomatis | 69 diganti EID (DOI tidak ditemukan & DOI benar tidak ada) | 9 DOI bisa dibuka tetapi metadata beda -> DOI asli dipertahankan | 0 ditunda (server tidak dapat dihubungi)
[8-merge] 4250 publikasi hasil merge (gabungan >1 baris) | total 10227 publikasi unik


## Tahap 10 — Tabel `bidang` dan `pub_bidang`
`bidang` = daftar bidang ilmu (dari `DAFTAR_FILE`, tanpa duplikasi nama). `pub_bidang` = relasi publikasi ↔ bidang: tiap publikasi
mendapat satu relasi untuk **setiap** file tempat ia ditemukan (publikasi yang sama di Physics dan Medicine = satu `publication_id`,
dua baris relasi). Baris prototype tidak punya bidang kecuali ditemukan juga di file CSV.

In [10]:
def tahap10_bidang(R):
    nama_bidang = [d["bidang"] for d in FILES]
    uniq = list(dict.fromkeys(nama_bidang, ))
    bidang = pd.DataFrame({"bidang_id": [f"BID{i:03d}" for i in range(1, len(uniq) + 1)], "nama_bidang": uniq})
    peta_file = {d["file"]: d["bidang"] for d in FILES}
    id_bidang = dict(zip(bidang["nama_bidang"], bidang["bidang_id"]))
    rel = (R[R["file_sumber"].isin(peta_file)][["publication_id", "file_sumber"]].drop_duplicates())
    rel["bidang_id"] = rel["file_sumber"].map(peta_file).map(id_bidang)
    pub_bidang = rel[["publication_id", "bidang_id"]].drop_duplicates().sort_values(["publication_id", "bidang_id"]).reset_index(drop=True)
    return bidang, pub_bidang

BIDANG, PUB_BIDANG = tahap10_bidang(R)
catat("10-bidang", f"{len(BIDANG)} bidang | {len(PUB_BIDANG)} relasi pub_bidang | "
                   f"{(PUB_BIDANG.groupby('publication_id').size() > 1).sum()} publikasi punya >1 bidang")

[10-bidang] 9 bidang | 15134 relasi pub_bidang | 4249 publikasi punya >1 bidang


## Tahap 11 — Tabel keseluruhan (+ tabel per CSV)
Dibangun dari publikasi unik (`PUB`): PUBLICATIONS, AUTHORS, PUB_AUTHOR, INSTITUTIONS, PUB_INSTITUTION, KEYWORDS, FUNDING,
CHUNKS, REFERENCES. `author` / `keyword` / `institusi` dipisah per `;`; ID institusi & keyword memakai ID prototype bila sudah ada.

In [11]:
def bangun_tabel(clean):
    """clean = 1 baris per publikasi unik (ada publication_id). Return dict tabel."""
    # --- ID acuan dari prototype (institusi, keyword, funding) supaya ID lama tidak berubah
    acu_inst, acu_kw, acu_fund, max_fnd = {}, {}, {}, 0
    if PROTOTYPE:
        acu_inst = {norm_key(n): i for i, n in zip(PROTOTYPE["INSTITUTIONS"]["institution_id"], PROTOTYPE["INSTITUTIONS"]["institution_name"])}
        acu_kw = {norm_key(k): i for i, k in zip(PROTOTYPE["KEYWORDS"]["keyword_id"], PROTOTYPE["KEYWORDS"]["keyword"])}
        f = PROTOTYPE["FUNDING"]
        acu_fund = {(p, str(t)): i for i, p, t in zip(f["funding_id"], f["publication_id"], f["source_text"])}
        max_fnd = max([int(re.sub(r"\D", "", i) or 0) for i in f["funding_id"]] + [0])
    clean = clean.copy()
    # PUBLICATIONS
    pub_cols = ["publication_id", "eid", "doi", "title", "year", "source_title", "volume", "issue", "art_no", "page_start", "page_end",
                "citation_count", "link", "abstract", "document_type", "publication_stage", "open_access", "issn",
                "language_of_original_document", "publisher", "source", "doi_status", "doi_asli"]
    for c in pub_cols:
        if c not in clean: clean[c] = None
    publications = clean[pub_cols].copy()
    # DOI tidak boleh kosong: jika publikasi tidak punya DOI, kolom doi diisi ID Scopus (EID) -> pencocokan tetap lewat EID
    _ganti = [not ada(d) for d in publications["doi"]]                                            # kosong -> EID
    publications["doi_status"] = [("kosong_diisi_eid" if g else st) for g, st in zip(_ganti, clean["doi_status"])]
    publications["doi"] = [e if g else d for d, e, g in zip(publications["doi"], publications["eid"], _ganti)]
    publications["search_text"] = ["\n\n".join(p for p in (t, a) if isinstance(p, str) and p) for t, a in zip(clean["title"], clean["abstract"])]
    # AUTHORS & PUB_AUTHOR
    pa_rows = []
    for r in clean.to_dict("records"):
        short, full, ids = split_semicolon(r["authors"]), split_semicolon(r["author_full_names"]), split_semicolon(r["author_s_id"])
        for i in range(max(len(short), len(full), len(ids))):
            m = FULL_RE.match(full[i]) if i < len(full) else None
            name = clean_text(m.group("name")) if m else (full[i] if i < len(full) else None)
            aid = ids[i] if i < len(ids) else (m.group("id") if m else None)
            if not name:
                name = short[i] if i < len(short) else None
            if not aid:
                aid = make_id("NOID", norm_key(name or f"{r['publication_id']}-{i}"))
            pa_rows.append((r["publication_id"], aid, i + 1, name))
    pa = pd.DataFrame(pa_rows, columns=["publication_id", "author_id", "author_order", "author_name"]).drop_duplicates(["publication_id", "author_id"])
    pub_author = pa[["publication_id", "author_id", "author_order"]].reset_index(drop=True)
    commonest = lambda s: (s.dropna().mode().iloc[0] if len(s.dropna()) else None)
    authors = pa.groupby("author_id", sort=False)["author_name"].agg(commonest).reset_index()
    authors["authors_name_normalized"] = authors["author_name"].str.strip().str.lower()
    # INSTITUTIONS & PUB_INSTITUTION
    inst_rows = [(r["publication_id"], acu_inst.get(norm_key(n), make_id("INS", norm_key(n))), n)
                 for r in clean.to_dict("records") for n in split_semicolon(r["affiliations"])]
    pi = pd.DataFrame(inst_rows, columns=["publication_id", "institution_id", "institution_name"])
    pub_institution = pi[["publication_id", "institution_id"]].drop_duplicates().reset_index(drop=True)
    institutions = pi.drop_duplicates("institution_id")[["institution_id", "institution_name"]].reset_index(drop=True)
    institutions["institution_name"] = institutions["institution_name"].map(capitalize_words)
    institutions["city"] = None
    institutions["country"] = institutions["institution_name"].map(guess_country)
    institutions["institutions_name_normalized"] = institutions["institution_name"].str.strip().str.lower()
    assert institutions["institution_id"].is_unique
    # KEYWORDS
    kw_rows = [(r["publication_id"], acu_kw.get(norm_key(k), make_id("KW", norm_key(k))), k, kt)
               for r in clean.to_dict("records") for kt, col in (("author", "author_keywords"), ("index", "index_keywords")) for k in split_semicolon(r[col])]
    keywords = (pd.DataFrame(kw_rows, columns=["publication_id", "keyword_id", "keyword", "keyword_type"])
                  .drop_duplicates(["publication_id", "keyword_id", "keyword_type"]).reset_index(drop=True))
    # FUNDING
    fund_rows = []
    for r in clean.to_dict("records"):
        for j, piece in enumerate(split_semicolon(r["funding_details"])):
            m = GRANT_RE.search(piece)
            fund_rows.append((r["publication_id"], (piece[:m.start()] if m else piece).strip(" ,;") or None,
                              m.group(1).strip() if m else None, r["funding_texts"] if j == 0 else None, piece))
    funding = pd.DataFrame(fund_rows, columns=["publication_id", "funding_agency", "grant_number", "funding_text", "source_text"])
    fids, nxt = [], max_fnd + 1
    for r in funding.itertuples():
        fid = acu_fund.get((r.publication_id, str(r.source_text)))
        if fid is None:
            fid = f"FND{nxt:06d}"; nxt += 1
        fids.append(fid)
    funding.insert(1, "funding_id", fids)
    assert funding["funding_id"].is_unique
    funding["funding_agency_normalized"] = funding["funding_agency"].str.strip().str.lower()
    # CHUNKS & REFERENCES
    ch = [(f"{r.publication_id}_CH{i:03d}", r.publication_id, "title_abstract", c, "abstract_metadata")
          for r in publications.itertuples() for i, c in enumerate(make_chunks(r.search_text, MAKS_KARAKTER_CHUNK), start=1)]
    chunks = pd.DataFrame(ch, columns=["chunk_id", "publication_id", "section", "chunk_text", "source_type"])
    rf = [(r["publication_id"], i, t) for r in clean.to_dict("records") for i, t in enumerate(split_references(r["references"], MODE_REFERENCES), start=1)]
    references = pd.DataFrame(rf, columns=["publication_id", "reference_order", "reference_text"])
    return {"PUBLICATIONS": publications, "AUTHORS": authors, "PUB_AUTHOR": pub_author, "INSTITUTIONS": institutions,
            "PUB_INSTITUTION": pub_institution, "KEYWORDS": keywords, "FUNDING": funding, "CHUNKS": chunks, "REFERENCES": references}

TABEL = bangun_tabel(PUB)
# CLEAN = SEMUA baris asli (termasuk yang digabung) dengan publication_id & file_sumber
_kolom_clean = [c for c in KOLOM if c != "eid"] + ["eid", "publication_id", "citation_count", "file_sumber"]
CLEAN = R.sort_values(["urut_sumber", "nomor_baris"])[_kolom_clean].reset_index(drop=True)
TABEL = {"RAW": RAW_SEMUA, "CLEAN": CLEAN, **TABEL, "BIDANG": BIDANG, "PUB_BIDANG": PUB_BIDANG}
catat("11-tabel", "tabel keseluruhan: " + ", ".join(f"{k}={len(v)}" for k, v in TABEL.items()))

# --- tabel per CSV: publikasi yang punya >=1 baris di file itu; ID sama dengan tabel keseluruhan ---
def tabel_per_file(nama):
    pids = set(R.loc[R["file_sumber"] == nama, "publication_id"])
    t = {"RAW": RAW_SEMUA[RAW_SEMUA["file_sumber"] == nama].reset_index(drop=True),
         "CLEAN": CLEAN[CLEAN["file_sumber"] == nama].reset_index(drop=True)}
    for k in ("PUBLICATIONS", "PUB_AUTHOR", "PUB_INSTITUTION", "KEYWORDS", "FUNDING", "CHUNKS", "REFERENCES", "PUB_BIDANG"):
        t[k] = TABEL[k][TABEL[k]["publication_id"].isin(pids)].reset_index(drop=True)
    t["AUTHORS"] = TABEL["AUTHORS"][TABEL["AUTHORS"]["author_id"].isin(t["PUB_AUTHOR"]["author_id"])].reset_index(drop=True)
    t["INSTITUTIONS"] = TABEL["INSTITUTIONS"][TABEL["INSTITUTIONS"]["institution_id"].isin(t["PUB_INSTITUTION"]["institution_id"])].reset_index(drop=True)
    t["BIDANG"] = TABEL["BIDANG"][TABEL["BIDANG"]["bidang_id"].isin(t["PUB_BIDANG"]["bidang_id"])].reset_index(drop=True)
    urut = ["RAW", "CLEAN", "PUBLICATIONS", "AUTHORS", "PUB_AUTHOR", "INSTITUTIONS", "PUB_INSTITUTION", "KEYWORDS", "FUNDING",
            "CHUNKS", "REFERENCES", "BIDANG", "PUB_BIDANG"]
    return {k: t[k] for k in urut}
TABEL_FILE = {d["file"]: tabel_per_file(d["file"]) for d in FILES}

[11-tabel] tabel keseluruhan: RAW=15139, CLEAN=15139, PUBLICATIONS=10227, AUTHORS=44989, PUB_AUTHOR=260436, INSTITUTIONS=10903, PUB_INSTITUTION=75764, KEYWORDS=162605, FUNDING=33272, CHUNKS=20039, REFERENCES=518294, BIDANG=9, PUB_BIDANG=15134


## Pemeriksaan konsistensi (primary key, foreign key, ID tanpa celah, keseragaman tabel keseluruhan ↔ per file)

In [12]:
def periksa_konsistensi():
    T = TABEL; P = T["PUBLICATIONS"]
    cek = {
        "publication_id unik (PUBLICATIONS)": P["publication_id"].is_unique,
        "doi tidak kosong di PUBLICATIONS (kosong -> diisi EID)": P["doi"].map(ada).all(),
        "publication_id berurutan tanpa celah": sorted(int(x[3:]) for x in P["publication_id"]) == list(range(1, len(P) + 1)),
        "CLEAN: semua baris sumber tercatat": len(T["CLEAN"]) == len(R),
        "CLEAN → publikasi valid": T["CLEAN"]["publication_id"].isin(P["publication_id"]).all(),
        "PUB_AUTHOR → publikasi & author valid": T["PUB_AUTHOR"]["publication_id"].isin(P["publication_id"]).all() and T["PUB_AUTHOR"]["author_id"].isin(T["AUTHORS"]["author_id"]).all(),
        "author_id unik": T["AUTHORS"]["author_id"].is_unique,
        "PUB_INSTITUTION → publikasi & institusi valid": T["PUB_INSTITUTION"]["publication_id"].isin(P["publication_id"]).all() and T["PUB_INSTITUTION"]["institution_id"].isin(T["INSTITUTIONS"]["institution_id"]).all(),
        "KEYWORDS/FUNDING/CHUNKS/REFERENCES → publikasi valid": all(T[k]["publication_id"].isin(P["publication_id"]).all() for k in ("KEYWORDS", "FUNDING", "CHUNKS", "REFERENCES")),
        "funding_id & chunk_id unik": T["FUNDING"]["funding_id"].is_unique and T["CHUNKS"]["chunk_id"].is_unique,
        "bidang_id & nama_bidang unik": T["BIDANG"]["bidang_id"].is_unique and T["BIDANG"]["nama_bidang"].is_unique,
        "PUB_BIDANG → publikasi & bidang valid": T["PUB_BIDANG"]["publication_id"].isin(P["publication_id"]).all() and T["PUB_BIDANG"]["bidang_id"].isin(T["BIDANG"]["bidang_id"]).all(),
        "PUB_BIDANG tanpa duplikat": not T["PUB_BIDANG"].duplicated().any(),
        "Tidak ada nama author mengandung ';'": not T["AUTHORS"]["author_name"].fillna("").str.contains(";").any(),
    }
    # tabel per file harus sama persis dengan tabel keseluruhan untuk ID yang sama
    sama = True
    for nama, t in TABEL_FILE.items():
        a = t["PUBLICATIONS"].set_index("publication_id").sort_index()
        b = P.set_index("publication_id").loc[a.index].sort_index()
        sama &= a.fillna("").astype(str).equals(b.fillna("").astype(str))
    cek["Tabel per CSV konsisten dengan tabel keseluruhan (ID & isi)"] = bool(sama)
    for k, v in cek.items():
        print("✔" if v else "✘", k)
    if not all(cek.values()):
        raise AssertionError("Pemeriksaan konsistensi gagal: " + ", ".join(k for k, v in cek.items() if not v))
    return cek

CEK = periksa_konsistensi()

✔ publication_id unik (PUBLICATIONS)
✔ doi tidak kosong di PUBLICATIONS (kosong -> diisi EID)
✔ publication_id berurutan tanpa celah
✔ CLEAN: semua baris sumber tercatat
✔ CLEAN → publikasi valid
✔ PUB_AUTHOR → publikasi & author valid
✔ author_id unik
✔ PUB_INSTITUTION → publikasi & institusi valid
✔ KEYWORDS/FUNDING/CHUNKS/REFERENCES → publikasi valid
✔ funding_id & chunk_id unik
✔ bidang_id & nama_bidang unik
✔ PUB_BIDANG → publikasi & bidang valid
✔ PUB_BIDANG tanpa duplikat
✔ Tidak ada nama author mengandung ';'
✔ Tabel per CSV konsisten dengan tabel keseluruhan (ID & isi)


## Tahap 12 — Laporan akhir, log, dan penyimpanan
Laporan: jumlah data awal per file, duplikat dalam file, duplikat antar-file, status DOI, publikasi yang berhasil di-merge,
publikasi unik, pasangan yang tidak digabung, dan jumlah publikasi per bidang.

In [13]:
def tahap12_laporan():
    per_file = []
    for nama, g in R.groupby("file_sumber", sort=False):
        n_klaster_dalam = g["klaster_dalam_file"].nunique()
        pids = set(g["publication_id"])
        lintas = R[R["publication_id"].isin(pids) & (R["file_sumber"] != nama)]["publication_id"].nunique()
        per_file.append({"file": nama, "bidang": dict((d["file"], d["bidang"]) for d in FILES).get(nama, "(data awal)"),
                         "data_awal_baris": len(g), "duplikasi_dalam_file_baris": len(g) - n_klaster_dalam,
                         "publikasi_unik_dalam_file": len(pids), "publikasi_juga_ada_di_file_lain": lintas})
    per_file = pd.DataFrame(per_file)
    status = pd.Series([status_doi(i) for i in R.index]).value_counts()
    doi_unik = R.dropna(subset=["doi_norm"]).drop_duplicates("doi_norm")
    status_unik = pd.Series([status_doi(i) for i in doi_unik.index]).value_counts()
    n_antar = int((R.groupby("publication_id")["file_sumber"].nunique() > 1).sum())
    n_merge = int((R.groupby("publication_id").size() > 1).sum())
    baris_dibuang = len(R) - R["publication_id"].nunique()
    ringkas = pd.DataFrame([
        ["Total baris awal (semua sumber)", len(R)],
        ["Duplikasi dalam file (baris yang digabung ke baris lain di file yang sama)", int(sum(per_file["duplikasi_dalam_file_baris"]))],
        ["Duplikasi antar-file (publikasi yang ditemukan di >1 file)", n_antar],
        ["Pasangan digabung otomatis: dalam file / antar-file", f"{STAT_TAHAP['dalam_file']['digabung']} / {STAT_TAHAP['antar_file']['digabung']}"],
        ["Publikasi hasil merge (gabungan >1 baris)", n_merge],
        ["Baris yang menjadi bagian merge (total baris - publikasi unik)", baris_dibuang],
        ["Publikasi unik setelah cleaning", R["publication_id"].nunique()],
        ["Peringatan: EID yang sama tetapi publication_id berbeda (EID bukan patokan merge)", int((R.dropna(subset=["eid"]).groupby("eid")["publication_id"].nunique() > 1).sum())],
        ["Pasangan kandidat yang TIDAK digabung (bukti kurang / beda publikasi)", sum(1 for e in KEPUTUSAN_LOG if e["keputusan"] != "auto")],
        ["DOI valid_cocok (baris)", int(status.get("valid_cocok", 0))],
        ["DOI valid_terdaftar (baris)", int(status.get("valid_terdaftar", 0))],
        ["DOI valid_tidak_cocok (baris)", int(status.get("valid_tidak_cocok", 0))],
        ["DOI tidak_dapat_diakses (baris)", int(status.get("tidak_dapat_diakses", 0))],
        ["DOI tidak_ditemukan (baris)", int(status.get("tidak_ditemukan", 0))],
        ["DOI format_salah (baris)", int(status.get("format_salah", 0))],
        ["DOI kosong (baris)", int(status.get("kosong", 0))],
        ["DOI belum divalidasi (hanya jika VALIDASI_DOI='kandidat') (baris)", int(status.get("belum_divalidasi", 0))],
        *[[f"Publikasi dengan doi_status = {k}", int(v)] for k, v in TABEL["PUBLICATIONS"]["doi_status"].value_counts().items()],
        ["Cakupan validasi DOI", VALIDASI_DOI + f" ({sum(1 for d in DOI_CACHE.values() if d.get('dicoba_run_ini'))} DOI diminta Run ini)"],
    ], columns=["item", "nilai"])
    per_bidang = (PUB_BIDANG.merge(BIDANG, on="bidang_id").groupby(["bidang_id", "nama_bidang"])["publication_id"].nunique()
                  .reset_index(name="jumlah_publikasi"))
    tanpa_bidang = len(set(PUB["publication_id"]) - set(PUB_BIDANG["publication_id"]))
    per_bidang.loc[len(per_bidang)] = ["-", "(tanpa bidang: data awal yang tidak ditemukan di file CSV)", tanpa_bidang]
    pp = TABEL["PUBLICATIONS"]
    bermasalah = pp[~pp["doi_status"].isin(["valid_cocok", "valid_terdaftar"])].copy()
    _auth = PUB.set_index("publication_id")["authors"].to_dict()
    def _diag(row):
        c = DOI_CACHE.get(norm_doi(row["doi_asli"]) if ada(row["doi_asli"]) else norm_doi(row["doi"]))
        if not c or c.get("status") != "ok":
            return pd.Series({"judul_menurut_doi": "", "tahun_menurut_doi": "", "skor_judul": None, "alasan": "metadata DOI tidak tersedia (" + (c["status"] if c else "tidak dicek") + ")"})
        n = nilai_cocok(row["title"], row["year"], _auth.get(row["publication_id"]), c)
        alasan = ", ".join(x for x, ok in (("judul beda", n["skor"] < AMBANG_JUDUL_LONGGAR), ("tahun beda", not n["tahun_ok"]), ("author beda", not n["author_ok"])) if ok) or "mendekati ambang"
        return pd.Series({"judul_menurut_doi": c["title"], "tahun_menurut_doi": c["year"], "skor_judul": round(n["skor"], 2), "alasan": alasan})
    bermasalah = pd.concat([bermasalah, bermasalah.apply(_diag, axis=1)], axis=1) if len(bermasalah) else bermasalah
    bermasalah = bermasalah.assign(link_doi=lambda d: [f"https://doi.org/{x}" if ada(x) and DOI_RE.match(x.lower()) else "" for x in d["doi"]])[
        [c for c in ["publication_id", "eid", "doi", "doi_asli", "doi_status", "alasan", "skor_judul", "title", "judul_menurut_doi", "year", "tahun_menurut_doi", "link_doi"] if c in bermasalah.columns or c == "link_doi"]]
    return {"RINGKASAN": ringkas, "PER_FILE": per_file, "KUALITAS_DATA": KUALITAS,
            "STATUS_DOI_UNIK": status_unik.rename_axis("status").reset_index(name="jumlah_doi_unik"),
            "PER_BIDANG": per_bidang, "DOI_BERMASALAH": bermasalah}

LAPORAN = tahap12_laporan()
for k in ("RINGKASAN", "PER_FILE", "PER_BIDANG", "STATUS_DOI_UNIK"):
    print(f"\n=== {k} ===")
    display(LAPORAN[k]) if "display" in globals() else print(LAPORAN[k].to_string(index=False))


=== RINGKASAN ===
                                                                             item                            nilai
                                                  Total baris awal (semua sumber)                            15139
       Duplikasi dalam file (baris yang digabung ke baris lain di file yang sama)                                5
                       Duplikasi antar-file (publikasi yang ditemukan di >1 file)                             4249
                              Pasangan digabung otomatis: dalam file / antar-file                         5 / 5609
                                        Publikasi hasil merge (gabungan >1 baris)                             4250
                   Baris yang menjadi bagian merge (total baris - publikasi unik)                             4912
                                                  Publikasi unik setelah cleaning                            10227
Peringatan: EID yang sama tetapi publication_id berbeda (EID 

### Pasangan kandidat yang tidak digabung
Hanya informasi (tidak perlu tindakan): kandidat duplikat yang **tidak** digabung beserta alasannya. Kalau Anda yakin ada pasangan
yang sebenarnya sama, isi EID-nya di `PAKSA_GABUNG` pada pengaturan.

In [14]:
def susun_log_keputusan():
    return pd.DataFrame([{"lingkup": e["lingkup"], "kunci": e.get("kunci"), "bukti": "+".join(sorted(e["bukti"])), "rasio_author": round(e["rasio"], 2),
                          "keputusan": "gabung" if e["keputusan"] == "auto" else "tidak digabung", "alasan": e["alasan"],
                          "eid_a": R.at[e["i"], "eid"], "eid_b": R.at[e["j"], "eid"], "doi_a": R.at[e["i"], "doi"], "doi_b": R.at[e["j"], "doi"],
                          "status_doi_a": status_doi(e["i"]), "status_doi_b": status_doi(e["j"]),
                          "title_a": R.at[e["i"], "title"], "title_b": R.at[e["j"], "title"],
                          "file_a": R.at[e["i"], "file_sumber"], "file_b": R.at[e["j"], "file_sumber"]} for e in KEPUTUSAN_LOG])
LOG_KEPUTUSAN = susun_log_keputusan()
TIDAK_DIGABUNG = LOG_KEPUTUSAN[LOG_KEPUTUSAN["keputusan"] == "tidak digabung"] if len(LOG_KEPUTUSAN) else LOG_KEPUTUSAN
print(f"{len(LOG_KEPUTUSAN)} pasangan kandidat: {len(LOG_KEPUTUSAN) - len(TIDAK_DIGABUNG)} digabung, {len(TIDAK_DIGABUNG)} tidak digabung")
if len(TIDAK_DIGABUNG):
    display(TIDAK_DIGABUNG.head(20)) if "display" in globals() else print(TIDAK_DIGABUNG.head(20).to_string())

5615 pasangan kandidat: 5614 digabung, 1 tidak digabung
      lingkup                                    kunci  bukti  rasio_author       keputusan                                                                                                                                       alasan                eid_a                eid_b                       doi_a                       doi_b     status_doi_a status_doi_b                                                                                                                                                   title_a                                                                                                                                                   title_b         file_a         file_b
5  dalam_file  2-s2.0-105006712187|2-s2.0-105020771983  judul           0.0  tidak digabung  tidak digabung [author]: author hanya 0% sama (DOI berbeda, digabung ke DOI yang terverifikasi (tidak_ditemukan / valid_cocok); judul sama)  2-s2.0-1050

### Simpan semua hasil
Struktur folder hasil (di dalam `NAMA_FOLDER/hasil_normalisasi`):
- `keseluruhan/` — tabel keseluruhan (CSV per tabel + 1 file Excel + `schema_postgres.sql` & `import_postgres.sql`)
- `per_file/<nama file>/` — tabel masing-masing CSV (Excel + folder CSV), ID sama dengan keseluruhan
- `laporan/` — laporan pemeriksaan, `log_merge.csv`, `log_perubahan_id.csv`, `log_proses.csv`, `cache_validasi_doi.csv`, `pemetaan_publication_id.csv`
- `data_mentah_backup/` — salinan CSV asli + `manifest_sha1.csv`
Hasil Run sebelumnya dipindahkan ke `cadangan_sebelumnya/` (hanya 1 versi terakhir).

In [15]:
TIPE_INT = {"year", "citation_count", "author_order", "reference_order"}
def ddl_postgres(tabel):
    """schema_postgres.sql dibuat dari kolom tabel yang sebenarnya, supaya selalu cocok dengan CSV."""
    pk = {"PUBLICATIONS": "publication_id", "AUTHORS": "author_id", "INSTITUTIONS": "institution_id", "FUNDING": "funding_id",
          "CHUNKS": "chunk_id", "BIDANG": "bidang_id"}
    pk_gab = {"PUB_AUTHOR": ("publication_id", "author_id"), "PUB_INSTITUTION": ("publication_id", "institution_id"),
              "PUB_BIDANG": ("publication_id", "bidang_id"), "REFERENCES": ("publication_id", "reference_order")}
    fk = {"PUB_AUTHOR": [("publication_id", "publications"), ("author_id", "authors")],
          "PUB_INSTITUTION": [("publication_id", "publications"), ("institution_id", "institutions")],
          "KEYWORDS": [("publication_id", "publications")], "FUNDING": [("publication_id", "publications")],
          "CHUNKS": [("publication_id", "publications")], "REFERENCES": [("publication_id", "publications")],
          "PUB_BIDANG": [("publication_id", "publications"), ("bidang_id", "bidang")]}
    urut = ["PUBLICATIONS", "AUTHORS", "INSTITUTIONS", "BIDANG", "PUB_AUTHOR", "PUB_INSTITUTION", "KEYWORDS", "FUNDING", "CHUNKS", "REFERENCES", "PUB_BIDANG"]
    q = lambda n: '"references"' if n == "REFERENCES" else n.lower()
    sql = ["-- Dibuat otomatis oleh notebook normalisasi. Jalankan file ini dulu, lalu import_postgres.sql.", ""]
    for n in urut:
        df = tabel[n]
        kol = [f"  {c} {'INTEGER' if c in TIPE_INT else 'TEXT'}{' NOT NULL' if (n, c) == ('PUBLICATIONS', 'doi') else ''}" for c in df.columns]
        if n in pk: kol.append(f"  PRIMARY KEY ({pk[n]})")
        if n in pk_gab: kol.append(f"  PRIMARY KEY ({', '.join(pk_gab[n])})")
        if n == "BIDANG": kol.append("  UNIQUE (nama_bidang)")
        for c, ref in fk.get(n, []):
            kol.append(f"  FOREIGN KEY ({c}) REFERENCES {ref}({c})")
        sql.append(f"CREATE TABLE IF NOT EXISTS {q(n)} (\n" + ",\n".join(kol) + "\n);\n")
    imp = ["-- Jalankan di psql dari folder 'keseluruhan' (urutan memenuhi foreign key).", ""]
    for n in urut:
        imp.append(f"\\copy {q(n)} FROM '{n.lower()}.csv' WITH (FORMAT csv, HEADER true, ENCODING 'UTF8');")
    return "\n".join(sql), "\n".join(imp) + "\n"

def simpan_semua():
    if not SIMPAN_HASIL:
        print("DRY RUN (SIMPAN_HASIL = False): tidak ada file ditulis."); return
    os.makedirs(HASIL_DIR, exist_ok=True)
    cad = os.path.join(HASIL_DIR, "cadangan_sebelumnya")
    shutil.rmtree(cad, ignore_errors=True)
    for sub in ("keseluruhan", "per_file"):                 # hasil Run sebelumnya -> cadangan
        lama = os.path.join(HASIL_DIR, sub)
        if os.path.isdir(lama):
            os.makedirs(cad, exist_ok=True); shutil.move(lama, os.path.join(cad, sub))
    os.makedirs(LAPORAN_DIR, exist_ok=True)

    readme = lambda judul, n, sumber: pd.DataFrame([
        ["Purpose", judul], ["Records", n], ["Source file", sumber],
        ["Publication ID rule", "satu publikasi = satu publication_id (merge berdasarkan DOI -> judul -> author; EID bukan patokan); tanpa celah"],
        ["Bidang", "tabel BIDANG + PUB_BIDANG (satu publikasi bisa punya banyak bidang)"],
        ["Dibuat", datetime.now().strftime("%Y-%m-%d %H:%M:%S")]], columns=["item", "value"])

    # keseluruhan
    kes = os.path.join(HASIL_DIR, "keseluruhan"); os.makedirs(kes, exist_ok=True)
    for n, df in TABEL.items():
        tulis_csv(df, os.path.join(kes, f"{n.lower()}.csv"))
    tulis_excel(os.path.join(kes, "scopus_research_intelligence_KESELURUHAN.xlsx"),
                {"README": readme("Scopus -> Research Intelligence (KESELURUHAN)", len(TABEL["PUBLICATIONS"]), ", ".join(([FILE_ACUAN] if FILE_ACUAN else []) + [d["file"] for d in FILES])), **TABEL})
    ddl, imp = ddl_postgres(TABEL)
    open(os.path.join(kes, "schema_postgres.sql"), "w", encoding="utf-8").write(ddl)
    open(os.path.join(kes, "import_postgres.sql"), "w", encoding="utf-8").write(imp)

    # per file
    for d in FILES:
        slug = re.sub(r"[^A-Za-z0-9]+", "_", os.path.splitext(d["file"])[0]).strip("_")
        pf = os.path.join(HASIL_DIR, "per_file", slug); os.makedirs(os.path.join(pf, "csv"), exist_ok=True)
        t = TABEL_FILE[d["file"]]
        for n, df in t.items():
            tulis_csv(df, os.path.join(pf, "csv", f"{n.lower()}.csv"))
        tulis_excel(os.path.join(pf, f"scopus_research_intelligence_{slug}.xlsx"),
                    {"README": readme(f"Scopus -> Research Intelligence ({d['bidang']})", len(t["PUBLICATIONS"]), d["file"]), **t})

    # laporan & log
    with pd.ExcelWriter(os.path.join(LAPORAN_DIR, "laporan_pemeriksaan.xlsx"), engine="openpyxl") as w:
        for k, df in LAPORAN.items():
            df.to_excel(w, sheet_name=k[:31], index=False)
        if len(TIDAK_DIGABUNG): safe_for_excel(TIDAK_DIGABUNG).to_excel(w, sheet_name="PASANGAN_TIDAK_DIGABUNG", index=False)
        gb = (PUB_BIDANG.merge(BIDANG, on="bidang_id").groupby("publication_id")["nama_bidang"].agg(", ".join).reset_index(name="bidang"))
        # tampilan baca saja (BUKAN untuk database): bidang digabung dengan koma
        safe_for_excel(PUB.merge(gb, on="publication_id", how="left")[["publication_id", "title", "year", "bidang"]]).to_excel(w, sheet_name="PUBLIKASI_PER_BIDANG", index=False)
    tulis_csv(LAPORAN["RINGKASAN"], os.path.join(LAPORAN_DIR, "laporan_pemeriksaan_ringkasan.csv"))
    tulis_csv(LOG_MERGE, os.path.join(LAPORAN_DIR, "log_merge.csv"))
    tulis_csv(PERUBAHAN_ID, os.path.join(LAPORAN_DIR, "log_perubahan_id.csv"))
    tulis_csv(LOG_KEPUTUSAN, os.path.join(LAPORAN_DIR, "log_keputusan_duplikat.csv"))
    tulis_csv(pd.DataFrame(LOG), os.path.join(LAPORAN_DIR, "log_proses.csv"))
    simpan_cache(); simpan_cache_koreksi()
    tulis_csv(R[["file_sumber", "nomor_baris", "kunci_eid", "eid", "doi", "title", "publication_id"]]
              .assign(status_doi=[status_doi(i) for i in R.index]), FILE_PEMETAAN)
    print("Hasil tersimpan di:", HASIL_DIR)

simpan_semua()

Hasil tersimpan di: /content/drive/MyDrive/NORMALISASI DATA BRIN/hasil_normalisasi


## (Opsional) Simpan ke database SQLite

In [ ]:
if SIMPAN_KE_SQLITE and SIMPAN_HASIL:
    from sqlalchemy import create_engine
    engine = create_engine(f"sqlite:///{os.path.join(HASIL_DIR, 'keseluruhan', 'scopus_ri.db')}")
    for name, df in TABEL.items():
        if name != "RAW":
            df.to_sql(name.lower(), engine, if_exists="replace", index=False)
    print("Database tersimpan di:", os.path.join(HASIL_DIR, "keseluruhan", "scopus_ri.db"))